### Vibrometer Measurements of Car
##### Measurement Summary:
- car off
- **car on**
- **car on, music playing**
- revving car engine

This notebook creates plots for when the car is on/not playing music and when it is on/playing music

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from scipy import signal
import scipy.stats as ss
from sklearn.preprocessing import MinMaxScaler
import scipy.fft
import pywt
from scipy.signal.windows import hann
from scipy import interpolate
from scipy.ndimage import uniform_filter
import ssqueezepy as sq
import subprocess
import os
from scipy.signal import welch
from ssqueezepy.experimental import scale_to_freq
from matplotlib.backends.backend_pdf import PdfPages

***
### Car on, no music
##### Load data:

In [ ]:
# Load data:
# (make sure the .ipynb file is in the same folder as the data)
# NOTE: SENSOR AIMED AT FRONT OF CAR (see V1_Photo.jpg)
column_names1 = ['Time1 (s)', 'Signal1 (m)']
df1 = pd.read_table("oncar1.txt", delimiter=r'\s+', encoding='latin-1', skiprows=5, names=column_names1)
#print(df1.head())

column_names2 = ['Time2 (s)', 'Signal2 (m)']
df2 = pd.read_table("caron2.txt", delimiter=r'\s+', encoding='latin-1', skiprows=5, names=column_names2)
#print(df2.head())

column_names3 = ['Time3 (s)', 'Signal3 (m)']
df3 = pd.read_table("caron3.txt", delimiter=r'\s+', encoding='latin-1', skiprows=5, names=column_names3)
#print(df3.head())

column_names4 = ['Time4 (s)', 'Signal4 (m)']
df4 = pd.read_table("caron4.txt", delimiter=r'\s+', encoding='latin-1', skiprows=5, names=column_names4)
#print(df4.head())

column_names5 = ['Time5 (s)', 'Signal5 (m)']
df5 = pd.read_table("caron5.txt", delimiter=r'\s+', encoding='latin-1', skiprows=5, names=column_names5)
#print(df5.head())
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# Then file naming convention changes
# NOTE: SENSOR AIMED AT SIDE OF CAR (near tire, see V2_photo.jpg)
column_names6 = ['Time6 (s)', 'Signal6 (m)']
df6 = pd.read_table("caronr1.txt", delimiter=r'\s+', encoding='latin-1', skiprows=5, names=column_names6)

column_names7 = ['Time7 (s)', 'Signal7 (m)']
df7 = pd.read_table("caronr2.txt", delimiter=r'\s+', encoding='latin-1', skiprows=5, names=column_names7)

column_names8 = ['Time8 (s)', 'Signal8 (m)']
df8 = pd.read_table("caronr3.txt", delimiter=r'\s+', encoding='latin-1', skiprows=5, names=column_names8)
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# Different naming convention
# NOTE: SENSOR AIMED AT FRONT OF CAR (greater distance, see V3_photo.jpg)
column_names9 = ['Time9 (s)', 'Signal9 (m)']
df9 = pd.read_table("caronw1.txt", delimiter=r'\s+', encoding='latin-1', skiprows=5, names=column_names9)

column_names10 = ['Time10 (s)', 'Signal10 (m)']
df10 = pd.read_table("caronw2.txt", delimiter=r'\s+', encoding='latin-1', skiprows=5, names=column_names10)

column_names11 = ['Time11 (s)', 'Signal11 (m)']
df11 = pd.read_table("caronw3.txt", delimiter=r'\s+', encoding='latin-1', skiprows=5, names=column_names11)

##### Plot Time Series:
Notes:
- dataframes 1-5: vibrometer sensor aimed at front of car
- dataframes 6-8: vibrometer sensor aimed at side of car
- dataframes 9-11: vibrometer sensor aimed at front of car, greater distance to car from sensor head

In [ ]:
car_on_dataframes = [df1, df2, df3, df4, df5, df6, df7, df8, df9, df10, df11] # add more after loading more dataframes
with PdfPages('raw_data_plots.pdf') as pdf:
    for i, dframe in enumerate(car_on_dataframes):
        time_column = f"Time{i+1} (s)"
        signal_column = f"Signal{i+1} (m)"
        # Plot:
        plt.figure(figsize=(12,4))
        plt.plot(dframe[time_column], dframe[signal_column], label=f"Car On Measurement {i+1} - Full Data")
        plt.xlabel("Time (s)")
        plt.ylabel("Amplitude (m)")
        plt.grid(True,alpha=0.5)
        plt.title(f"Car On Measurement {i+1} - Full Data")
        plt.tight_layout()
        pdf.savefig() # save figures as a pdf file
        plt.show()
        plt.close()

##### Detrending and Shifting Data to Zero Mean:

In [ ]:
with PdfPages('detrended_shifted_plots.pdf') as pdf:
    for i, dframe in enumerate(car_on_dataframes):
        time_column = f"Time{i+1} (s)"
        signal_column = f"Signal{i+1} (m)"
        detrended_signal_column = f"Signal{i+1} Detrended (m)"
        mean_shifted_column = f"Signal{i+1} Zero Mean (m)"
        
        # Detrend
        dframe[detrended_signal_column] = signal.detrend(dframe[signal_column], type='linear') 
        # Shift Mean to ~Zero (without normalizing)
        dframe[mean_shifted_column] = dframe[detrended_signal_column] - dframe[detrended_signal_column].mean()
    
        # Plot:
        plt.figure(figsize=(12,4))
        plt.plot(dframe[time_column], dframe[mean_shifted_column], label=f"Car On Measurement {i+1} - Full Data")
        plt.xlabel("Time (s)")
        plt.ylabel("Amplitude (m)")
        plt.grid(True,alpha=0.5)
        plt.axhline(y=0.0, label="0.0", color='red', linestyle=":")
        plt.title(f"Car On Measurement {i+1} - Detrended and Shifted to Zero Mean")
        plt.tight_layout()
        pdf.savefig()
        plt.show()
        plt.close()

##### Tapering Detrended and Shifted (DS) Time Series
- A Hann window is used for tapering

In [ ]:
# UPDATED TAPERING FUNCTION:
def apply_hann_window(data, column, taper_fraction = 0.1):
    """Taper edges of time series with Hann window
    taper_fraction = 0.1 will start taper at 10% from edges"""
    n= len(data)
    taper_length = int(n * taper_fraction)
    
    # Create a copy of the data
    windowed_data = data[column].copy()
    windowed_data = windowed_data.astype(float) # needed for FFT code to work properly
    
    # Apply Hann window to the left edge
    left_window = np.hanning(2 * taper_length)[:taper_length]
    windowed_data.iloc[:taper_length] *= left_window
    
    # Apply Hann window to the right edge
    right_window = np.hanning(2 * taper_length)[taper_length:]
    windowed_data.iloc[-taper_length:] *= right_window
    
    return windowed_data

with PdfPages('tapered_plots.pdf') as pdf:
    for i, dframe in enumerate(car_on_dataframes):
        tapered_data = []
        mean_shifted_column = f"Signal{i+1} Zero Mean (m)"
        hann_result = apply_hann_window(dframe, mean_shifted_column) # taper all dataframes
        tapered_data.append(hann_result)
        time_column = f"Time{i+1} (s)" # need time column for plotting
        # Plot:
        plt.figure(figsize=(12,4))
        plt.plot(dframe[time_column], dframe[mean_shifted_column], label=f"Full Data")
        plt.plot(dframe[time_column], tapered_data[-1], label=f"Tapered Data")
        plt.xlabel("Time (s)")
        plt.ylabel("Amplitude (m)")
        plt.grid(True,alpha=0.5)
        plt.title(f"Car On Measurement {i+1} - Full and Tapered DS Data")
        plt.legend()
        plt.tight_layout()
        pdf.savefig()
        plt.show()
        plt.close()

##### Subsampling DS Time Series

In [ ]:
# Subsampling the time series
#Sampling Rate = Total Number of Samples / Total Time (in seconds)
def subsample(data, column, decimation_factor):
    """Subsample data using scipy.signal.decimate with anti-aliasing filter.
    data: input data e.g. df1
    column: column to subsample (e.g. "Signal1 Zero Mean (m)"), string
    decimation_factor: factor by which to reduce sampling rate"""
    num_samples=len(data[column])
    sampling_time=data.iloc[-1, 0] #get last value in first column (time (s))
    original_sampling_rate = int(num_samples/sampling_time)
    print(f"Original sampling rate for {column}: {original_sampling_rate} Hz")
    # scipy.signal.decimate applies an anti-aliasing filter automatically
    # Use a higher order filter for better anti-aliasing (default is 8)
    decimated_data = scipy.signal.decimate(data[column], decimation_factor, ftype='iir', zero_phase=True) 
    # get new sampling rate:
    new_sr = int(original_sampling_rate / decimation_factor)
    print(f"New sampling rate for {column}: {new_sr} Hz")
    return decimated_data, new_sr, original_sampling_rate

with PdfPages('subsampled_data_plots.pdf') as pdf:
    for i, dframe in enumerate(car_on_dataframes):
        mean_shifted_column = f"Signal{i+1} Zero Mean (m)"
        time_column = f"Time{i+1} (s)"
        subsample_result = subsample(dframe, mean_shifted_column, decimation_factor=50)
        subsampled_data = subsample_result[0]
        # sub_list = []
        # sub_list.append(subsampled_data)
        
        # Plot:
        plt.figure(figsize=(12,4))
        plt.plot(dframe[time_column], dframe[mean_shifted_column], label=f"Full Data")
        plt.plot(dframe[time_column][::50], subsampled_data, label=f"Subsampled Data") # Make sure time step is same as decimation_factor
        plt.xlabel("Time (s)")
        plt.ylabel("Amplitude (m)")
        plt.grid(True, alpha=0.5)
        plt.title(f"Car On Measurement {i+1} - Full and Subsampled DS Data")
        plt.legend()
        #plt.xlim(4,5)
        plt.tight_layout()
        pdf.savefig()
        plt.show()
        plt.close()

##### Subsampling the Tapered DS Time Series

In [ ]:
def sub_tapered(tapered_data, decimation_factor):
    sub_tap_data = scipy.signal.decimate(tapered_data, decimation_factor, ftype='iir', zero_phase=True)
    return sub_tap_data

with PdfPages('subsampled_tapered_plots.pdf') as pdf:
    for i, dframe in enumerate(car_on_dataframes):
        mean_shifted_column = f"Signal{i+1} Zero Mean (m)"
        time_column = f"Time{i+1} (s)"
        sub_tap_result = sub_tapered(apply_hann_window(dframe, mean_shifted_column), decimation_factor=50)
        # PLot:
        plt.figure(figsize=(12,4))
        plt.plot(dframe[time_column], dframe[mean_shifted_column], label=f"Full Data")
        plt.plot(dframe[time_column][::50], sub_tap_result, label=f"Subsampled/Tapered Data") # Make sure time step is same as decimation_factor
        plt.xlabel("Time (s)")
        plt.ylabel("Amplitude (m)")
        plt.grid(True, alpha=0.5)
        plt.title(f"Car On Measurement {i+1} - Full and Subsampled/Tapered DS Data")
        plt.legend()
        plt.tight_layout()
        pdf.savefig()
        plt.show()
        plt.close()

##### Fast Fourier Transform
- Plotting FFT spectra showing tapered/subsampled data overlayed onto full DS data for a given dataframe (from original FFT code)
- Coherent gain calculated based on partial taper

In [ ]:
def calculate_coherent_gain(data, taper_fraction=0.1):
    """Calculate coherent gain for partial Hann taper"""
    n=len(data)
    taper_length = int(n * taper_fraction)
    
    # Create the full window
    window = np.ones(n)
    
    # Left taper
    left_window = np.hanning(2 * taper_length)[:taper_length]
    window[:taper_length] = left_window
    
    # Right taper
    right_window = np.hanning(2 * taper_length)[taper_length:]
    window[-taper_length:] = right_window
    
    # Coherent gain is the mean
    coherent_gain = np.mean(window)
    
    return coherent_gain

In [ ]:
# Plotting FFT spectra showing tapered/subsampled data overlayed onto full DNS data for a given dataframe
# def plot_fft(data, column, sr, fig1=None, fig2=None, fig3=None, name='', apply_window=True):
#     """Plot FFT with windowing correction.
#     data: array
#     sr: Sampling rate
#     apply_window : bool, says whether to apply Hann window (default True)"""
#     n = len(data)
    
#     # Apply window if requested
#     if apply_window:
#         windowed_data = apply_hann_window1(data, column, taper_fraction=0.1)
        
#         # Calculate coherent gain for partial taper
#         taper_length = int(n * 0.1)
#         window = np.ones(n)
#         left_window = np.hanning(2 * taper_length)[:taper_length]
#         window[:taper_length] = left_window
#         right_window = np.hanning(2 * taper_length)[taper_length:]
#         window[-taper_length:] = right_window
#         coherent_gain = np.mean(window)
        
#     else:
#         windowed_data = data[column].values
#         coherent_gain = 1.0
    
#     # Compute FFT
#     fft_values = scipy.fft.rfft(windowed_data)
#     freqs = scipy.fft.rfftfreq(n, d=1/sr)
    
#     # magnitudes WITH coherent gain correction
#     magnitudes = np.abs(fft_values) / (n * coherent_gain)
    
#     # Double the AC components (not DC and Nyquist)
#     magnitudes[1:] *= 2.0
    
#     # Undo doubling for Nyquist if even length
#     if n % 2 == 0:
#         magnitudes[-1] /= 2.0
    
#     phases = np.angle(fft_values)
    
#     # Create figures:
#     if fig1 is None:
#         fig1 = plt.figure(figsize=(10,6))
#         plt.plot(freqs, magnitudes, linewidth=1, label=name)
#         plt.title('FFT Magnitude Spectrum (Linear)', fontsize=16)
#         plt.xlabel('Frequency (Hz)', fontsize=14)
#         plt.ylabel('Magnitude', fontsize=14)
#         plt.grid(True, which='major', alpha=0.5)
#         plt.grid(True, which='minor', alpha=0.5)
#         plt.minorticks_on()
#         plt.xticks(fontsize=14)
#         plt.yticks(fontsize=14)
#         plt.tight_layout()
#         if name:
#             plt.legend(loc='upper right')
#     else:
#         plt.figure(fig1.number)
#         plt.plot(freqs, magnitudes, linewidth=1, label=name)
#         if name:
#             plt.legend(loc='upper right')
    
#     if fig2 is None:
#         fig2 = plt.figure(figsize=(10, 6))
#         plt.loglog(freqs[1:], magnitudes[1:], linewidth=1, label=name)
#         plt.title('FFT Magnitude Spectrum (log)', fontsize=16)
#         plt.xlabel('Frequency (Hz)', fontsize=14)
#         plt.ylabel('Magnitude', fontsize=14)
#         plt.xticks(fontsize=14)
#         plt.yticks(fontsize=14)
#         plt.grid(True, which='major', alpha=0.5)
#         plt.grid(True, which='minor', alpha=0.5)
#         plt.tight_layout()
#         if name:
#             plt.legend(loc='upper right')
#     else:
#         plt.figure(fig2.number)
#         plt.loglog(freqs[1:], magnitudes[1:], linewidth=1, label=name)
#         plt.legend()
#         if name:
#             plt.legend(loc='upper right')
    
#     if fig3 is None:
#         fig3 = plt.figure(figsize=(10, 6))
#         plt.plot(freqs[1:], phases[1:], linewidth=1, label=name)
#         plt.title('Phase Spectrum', fontsize=16)
#         plt.xlabel('Frequency (Hz)', fontsize=14)
#         plt.ylabel('Phase (radians)', fontsize=14)
#         plt.grid(True, alpha=0.5)
#         plt.xticks(fontsize=14)
#         plt.yticks(fontsize=14)
#         #plt.xlim(-1, 50)
#         plt.tight_layout()
#         if name:
#             plt.legend(loc='lower right', framealpha=0.5)
#     else:
#         plt.figure(fig3.number)
#         plt.plot(freqs[1:], phases[1:], linewidth=1, label=name)
#         if name:
#             plt.legend(loc='lower right', framealpha=0.8)
    
#     return fig1, fig2, fig3
# #~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# for i, dframe in enumerate(car_on_dataframes):
#     mean_shifted_column = f"Signal{i+1} Zero Mean (m)"
#     time_column = f"Time{i+1} (s)"
#     decimation_factor=50 # step used for subsampling
#     print(f"dataframe: df{i+1}")
#     # For full data
#     fig1, fig2, fig3 = plot_fft(dframe, mean_shifted_column, sr=25000, name='Full Data', apply_window=False)
#     # For tapered/subsampled data
#     sub_tap_results=[]
#     sub_tap_result = sub_tapered(apply_hann_window(dframe, mean_shifted_column), decimation_factor=50)
#     sub_tap_result = pd.DataFrame(sub_tap_result, columns=['tapered/subsampled'])
#     sub_tap_results.append(sub_tap_result)
#     plot_fft(sub_tap_results[-1], 'tapered/subsampled', sr=25000//decimation_factor, fig1=fig1, fig2=fig2, fig3=fig3, 
#              name='Subsampled/Tapered Data', apply_window=False)
#     plt.show()
#     plt.close()

##### Updated FFT With Reduced Frequency Resolution:
- Averaged frequency spectrum

In [ ]:
def plot_averaged_fft(data, column, sr, segment_size, title=''):
    """Plot FFT with frequency bin averaging 
    data: pandas DataFrame
    column: relevant data column from data, string
    sr: Sampling rate in Hz
    segment_size:segment size for averaging, int
    title: plot title, str"""
    # Goal: decrease frequency resolution of the FFT
    # Signal parameters
    T = 1 / sr  # Sampling period
    # signal x:
    x = data[column].values
    N = len(x)
    
    # Parameters for averaging
    overlap = 0.5   # 50% overlap
    step_size = int(segment_size * (1 - overlap))
    
    # Calculate number of segments
    num_segments = (N - segment_size) // step_size + 1
    
    # Initialize array to store summed FFT results
    avg_fft_magnitude = np.zeros(segment_size // 2 + 1) # only need one side for real input
    
    for i in range(num_segments):
        start = i * step_size
        end = start + segment_size
        segment = x[start:end]
        
        # apply a Hanning window to the segment to reduce spectral leakage
        window = np.hanning(segment_size)
        windowed_segment = segment * window
        
        # Compute FFT of the segment
        segment_fft = scipy.fft.rfft(windowed_segment, segment_size)
        
        # Get the magnitude of the positive frequency side
        #segment_magnitude = np.abs(segment_fft)[:segment_size // 2 + 1]
        segment_magnitude = np.abs(segment_fft)[:segment_size // 2 + 1] / np.sum(window) # window sum is for amplitude correction
        
        # Accumulate the magnitudes
        #avg_fft_magnitude += segment_magnitude 
        avg_fft_magnitude += segment_magnitude ** 2 #accumulate power
    
    # Calculate the mean magnitude
    #avg_fft_magnitude /= num_segments
    avg_fft_magnitude = np.sqrt(avg_fft_magnitude / num_segments) # don't take square root if want power spectral density instead
    
    # Get the frequency bins
    freqs = scipy.fft.rfftfreq(segment_size, T)[:segment_size // 2 + 1]
    
    # Plot the averaged frequency spectrum
    plt.figure(figsize=(12, 6))
    #plt.plot(freqs, avg_fft_magnitude)
    plt.loglog(freqs, avg_fft_magnitude)
    plt.xlabel('Frequency (Hz)', fontsize=12)
    plt.ylabel('Averaged FFT Magnitude', fontsize=12)
    plt.title(title)
    plt.grid(True, which='both', alpha=0.5)
    plt.ylim(bottom=10**-11)
    pdf.savefig()
    plt.show()
    plt.close()

    return freqs, avg_fft_magnitude
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
with PdfPages('Averaged_FFT_plots.pdf') as pdf:
    for i, dframe in enumerate(car_on_dataframes):
        mean_shifted_column = f"Signal{i+1} Zero Mean (m)"
        title_ = f"Averaged Frequency Spectrum - Measurement {i+1}"
        freqs, avg_mags = plot_averaged_fft(dframe, mean_shifted_column, sr=25000, segment_size=8096, title=title_)

##### Power Spectral Density With Welch's Method:
- Change the scaling parameter in welch() from 'density' to 'spectrum' to plot a squared magnitude spectrum instead

In [ ]:
def welch_psd(data, column, sr, title, segment_size, window='hann'):
    """data: pandas DataFrame
    column: relevant data column from data, str
    sr: Sampling rate in Hz
    title: title for PSD plot, str
    segment_size: length of each segment
    window: desired window to use"""
    
    overlap = segment_size//2 # number of points to overlap between segments, here is 50% overlap
    freqs, psd = welch(data[column].values, fs=sr, nperseg=segment_size, noverlap=overlap, window=window, scaling='density')

    # Plot:
    plt.figure(figsize=(12,6))
    #plt.plot(freqs,psd)
    plt.loglog(freqs, psd)
    plt.xlabel("Frequency (Hz)")
    plt.ylabel("PSD ($m^2$/ Hz)")
    plt.title(title)
    plt.grid(True, which='both', alpha=0.5)
    plt.ylim(bottom=10**-22)
    pdf.savefig()
    plt.show()
    plt.close()

    return freqs, psd
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
with PdfPages('PSD_plots.pdf') as pdf:
    for i, dframe in enumerate(car_on_dataframes):
        mean_shifted_column = f"Signal{i+1} Zero Mean (m)"
        w_title = f"Power Spectral Density - Measurement {i+1}"
        w_freqs, w_psd = welch_psd(dframe, mean_shifted_column, sr=25000, title=w_title, segment_size=8096)

##### Continuous Wavelet Transform
- The Morlet wavelet, given by $\psi(t)=\exp(\frac{-t^2}{2})\cos(5t)$, is used for the wavelet transform
- Note: may want to reduce look of horizontal bands near bottom of plots in future

In [ ]:
# Approach 2: ssqueezepy
with PdfPages('CWT_plots.pdf') as pdf:
    for i, dframe in enumerate(car_on_dataframes):
        # Setup Data:
        sampling_rate= 25000 # Hz
        sig_col= f"Signal{i+1} Zero Mean (m)"
        signal = apply_hann_window(dframe, sig_col)
        time = dframe[f"Time{i+1} (s)"]
    
        downsample_factor = 10 # downsample to speed up computation
        signal_ds = scipy.signal.decimate(signal, downsample_factor)
        sampling_rate_ds = sampling_rate // downsample_factor
        time_ds = time[::downsample_factor]
        print(f"Sample rate after downsampling: {sampling_rate_ds} Hz")
        print(f"Nyquist: {sampling_rate_ds / 2} Hz")
    
        # Setup Resolution:
        nv = 32 # number of voices/wavelets per octave
        # Perform CWT:
        Wx, scales = sq.cwt(signal_ds, 'morlet', nv=nv, fs=sampling_rate_ds)
        frequencies = scale_to_freq(scales, 'morlet', len(signal_ds), fs=sampling_rate_ds)
    
        # Max frequency should not exceed Nyquist --> limit freq range based on this:
        freq_mask = (frequencies >=5) & (frequencies <= sampling_rate_ds / 2)
        freqs_plot = frequencies[freq_mask]
        Wx_masked = Wx[freq_mask, :]         
    
        # Time downsample for display
        time_skip = 50
        Wx_plot = np.log10(np.abs(Wx_masked[:, ::time_skip])+ 1e-10) # add small epsilon to prevent log error
        time_display = time_ds[::time_skip]
    
        # Option: Define the "floor" for the data
        # e.g. anything quieter than threshold_val will be treated as "Not a Number" (NaN), which renders as transparent/white.
        threshold_val = -9.5 # note: vmin should be close to threshold value
        Wx_plot[Wx_plot < threshold_val] = np.nan
    
        # Plot:
        plt.figure(figsize=(14, 6))
        mesh = plt.pcolormesh(time_display, freqs_plot, Wx_plot, cmap='jet', shading='auto', vmax=-5.5, vmin=threshold_val)
        plt.yscale('log')
        plt.ylabel('Frequency (Hz)')
        plt.xlabel('Time (s)')
        plt.title(f'Car On Measurement {i+1}')
        # Add colorbar
        plt.colorbar(mesh, label='log\u2081\u2080(Magnitude)')
        pdf.savefig()
        plt.show()
        plt.close()

***
### Car on, music playing
##### Load data:

In [ ]:
# NOTE: SENSOR AIMED AT FRONT OF CAR (see V1_Photo.jpg)
# Load data:
columns_music1 = ['Time1 (s)', 'Signal1 (m)']
dfm1 = pd.read_table("musiccar1.txt", delimiter=r'\s+', encoding='latin-1', skiprows=5, names=columns_music1)
#print(dfm1.head())

columns_music2 = ['Time2 (s)', 'Signal2 (m)']
dfm2 = pd.read_table("musiccar2.txt", delimiter=r'\s+', encoding='latin-1', skiprows=5, names=columns_music2)
#print(dfm2.head())

columns_music3 = ['Time3 (s)', 'Signal3 (m)']
dfm3 = pd.read_table("musiccar3.txt", delimiter=r'\s+', encoding='latin-1', skiprows=5, names=columns_music3)
#print(dfm3.head())

##### Plot Time Series:
Notes:
- All measurements with music playing taken with the sensor aimed at the front of the car

In [ ]:
car_on_music_dataframes = [dfm1, dfm2, dfm3] 
with PdfPages('raw_data_plots_.pdf') as pdf:
    for i, dframe in enumerate(car_on_music_dataframes):
        time_column = f"Time{i+1} (s)"
        signal_column = f"Signal{i+1} (m)"
        # Plot:
        plt.figure(figsize=(12,4))
        plt.plot(dframe[time_column], dframe[signal_column], label=f"Car On Playing Music Measurement {i+1} - Full Data")
        plt.xlabel("Time (s)")
        plt.ylabel("Amplitude (m)")
        plt.grid(True, alpha=0.5)
        plt.title(f"Car On Playing Music Measurement {i+1} - Full Data")
        plt.tight_layout()
        pdf.savefig()
        plt.show()
        plt.close()

##### Detrending and Shifting Data to Zero Mean:

In [ ]:
with PdfPages('detrended_shifted_plots_.pdf') as pdf:
    for i, dframe in enumerate(car_on_music_dataframes):
        time_column = f"Time{i+1} (s)"
        signal_column = f"Signal{i+1} (m)"
        detrended_signal_column = f"Signal{i+1} Detrended (m)"
        mean_shifted_column = f"Signal{i+1} Zero Mean (m)"
        
        # Detrend
        dframe[detrended_signal_column] = scipy.signal.detrend(dframe[signal_column], type='linear') 
        # Shift Mean to ~Zero (without normalizing)
        dframe[mean_shifted_column] = dframe[detrended_signal_column] - dframe[detrended_signal_column].mean()
    
        # Plot:
        plt.figure(figsize=(12,4))
        plt.plot(dframe[time_column], dframe[mean_shifted_column], label=f"Car On Playing Music Measurement {i+1} - Full Data")
        plt.xlabel("Time (s)")
        plt.ylabel("Amplitude (m)")
        plt.grid(True,alpha=0.5)
        plt.axhline(y=0.0, label="0.0", color='red', linestyle=":")
        plt.title(f"Car On Playing Music Measurement {i+1} - Detrended and Shifted to Zero Mean")
        plt.tight_layout()
        pdf.savefig()
        plt.show()
        plt.close()

##### Tapering Detrended and Shifted (DS) Time Series
- A Hann window is used for tapering

In [ ]:
with PdfPages('tapered_plots_.pdf') as pdf:
    for i, dframe in enumerate(car_on_music_dataframes):
        tapered_data = []
        mean_shifted_column = f"Signal{i+1} Zero Mean (m)"
        hann_result = apply_hann_window(dframe, mean_shifted_column) # taper all dataframes
        tapered_data.append(hann_result)
        time_column = f"Time{i+1} (s)" # need time column for plotting
        # Plot:
        plt.figure(figsize=(12,4))
        plt.plot(dframe[time_column], dframe[mean_shifted_column], label=f"Full Data")
        plt.plot(dframe[time_column], tapered_data[-1], label=f"Tapered Data")
        plt.xlabel("Time (s)")
        plt.ylabel("Amplitude (m)")
        plt.grid(True,alpha=0.5)
        plt.title(f"Car On Playing Music Measurement {i+1} - Full and Tapered DS Data")
        plt.legend()
        plt.tight_layout()
        pdf.savefig()
        plt.show()
        plt.close()

##### Subsampling DS Time Series

In [ ]:
# Subsampling the tapered time series
#Sampling Rate = Total Number of Samples / Total Time (in seconds)
def subsample(data, column, decimation_factor):
    """Subsample data using scipy.signal.decimate with anti-aliasing filter.
    data: input data e.g. df1
    column: column to subsample (e.g. "Signal1 Zero Mean (m)"), string
    decimation_factor: factor by which to reduce sampling rate"""
    num_samples=len(data[column])
    sampling_time=data.iloc[-1, 0] #get last value in first column (time (s))
    original_sampling_rate = int(num_samples/sampling_time)
    print(f"Original sampling rate for {column}: {original_sampling_rate} Hz")
    # scipy.signal.decimate applies an anti-aliasing filter automatically
    # Use a higher order filter for better anti-aliasing (default is 8)
    decimated_data = scipy.signal.decimate(data[column], decimation_factor, ftype='iir', zero_phase=True) # 
    # get new sampling rate:
    new_sr = int(original_sampling_rate / decimation_factor)
    print(f"New sampling rate for {column}: {new_sr} Hz")
    return decimated_data, new_sr, original_sampling_rate

with PdfPages('subsampled_data_plots_.pdf') as pdf:
    for i, dframe in enumerate(car_on_music_dataframes):
        mean_shifted_column = f"Signal{i+1} Zero Mean (m)"
        time_column = f"Time{i+1} (s)"
        subsample_result = subsample(dframe, mean_shifted_column, decimation_factor=50)
        subsampled_data = subsample_result[0]
        # sub_list = []
        # sub_list.append(subsampled_data)
        
        # Plot:
        plt.figure(figsize=(12,4))
        plt.plot(dframe[time_column], dframe[mean_shifted_column], label=f"Full Data")
        plt.plot(dframe[time_column][::50], subsampled_data, label=f"Subsampled Data") # Make sure time step is same as decimation_factor
        plt.xlabel("Time (s)")
        plt.ylabel("Amplitude (m)")
        plt.grid(True, alpha=0.5)
        plt.title(f"Car On Playing Music Measurement {i+1} - Full and Subsampled DS Data")
        plt.legend()
        #plt.xlim(4,5)
        plt.tight_layout()
        pdf.savefig()
        plt.show()
        plt.close()

##### Subsampling the Tapered DS Time Series

In [ ]:
def sub_tapered(tapered_data, decimation_factor):
    sub_tap_data = scipy.signal.decimate(tapered_data, decimation_factor, ftype='iir', zero_phase=True)
    return sub_tap_data

with PdfPages('subsampled_tapered_plots_.pdf') as pdf:
    for i, dframe in enumerate(car_on_music_dataframes):
        mean_shifted_column = f"Signal{i+1} Zero Mean (m)"
        time_column = f"Time{i+1} (s)"
        sub_tap_result = sub_tapered(apply_hann_window(dframe, mean_shifted_column), decimation_factor=50)
        
        # PLot:
        plt.figure(figsize=(12,4))
        plt.plot(dframe[time_column], dframe[mean_shifted_column], label=f"Full Data")
        plt.plot(dframe[time_column][::50], sub_tap_result, label=f"Subsampled/Tapered Data") # Make sure time step is same as decimation_factor
        plt.xlabel("Time (s)")
        plt.ylabel("Amplitude (m)")
        plt.grid(True, alpha=0.5)
        plt.title(f"Car On Playing Music Measurement {i+1} - Full and Subsampled/Tapered DS Data")
        plt.legend()
        plt.tight_layout()
        pdf.savefig()
        plt.show()
        plt.close()

##### Updated FFT With Reduced Frequency Resolution:
- Averaged frequency spectrum

In [ ]:
def plot_averaged_fft(data, column, sr, segment_size, title=''):
    """Plot FFT with frequency bin averaging 
    data: pandas DataFrame
    column: relevant data column from data, string
    sr: Sampling rate in Hz
    segment_size:segment size for averaging, int
    title: plot title, str"""
    # Goal: decrease frequency resolution of the FFT
    # Signal parameters
    T = 1 / sr  # Sampling period
    # signal x:
    x = data[column].values
    N = len(x)
    
    # Parameters for averaging
    overlap = 0.5   # 50% overlap
    step_size = int(segment_size * (1 - overlap))
    
    # Calculate number of segments
    num_segments = (N - segment_size) // step_size + 1
    
    # Initialize array to store summed FFT results
    avg_fft_magnitude = np.zeros(segment_size // 2 + 1) # only need one side for real input
    
    for i in range(num_segments):
        start = i * step_size
        end = start + segment_size
        segment = x[start:end]
        
        # apply a Hanning window to the segment to reduce spectral leakage
        window = np.hanning(segment_size)
        windowed_segment = segment * window
        
        # Compute FFT of the segment
        segment_fft = scipy.fft.rfft(windowed_segment, segment_size)
        
        # Get the magnitude of the positive frequency side
        #segment_magnitude = np.abs(segment_fft)[:segment_size // 2 + 1]
        segment_magnitude = np.abs(segment_fft)[:segment_size // 2 + 1] / np.sum(window) # window sum is for amplitude correction
        
        # Accumulate the magnitudes
        #avg_fft_magnitude += segment_magnitude 
        avg_fft_magnitude += segment_magnitude ** 2 #accumulate power
    
    # Calculate the mean magnitude
    #avg_fft_magnitude /= num_segments
    avg_fft_magnitude = np.sqrt(avg_fft_magnitude / num_segments) # don't take square root if want power spectral density instead
    
    # Get the frequency bins
    freqs = scipy.fft.rfftfreq(segment_size, T)[:segment_size // 2 + 1]
    
    # Plot the averaged frequency spectrum
    plt.figure(figsize=(12, 6))
    #plt.plot(freqs, avg_fft_magnitude)
    plt.loglog(freqs, avg_fft_magnitude)
    plt.xlabel('Frequency (Hz)', fontsize=12)
    plt.ylabel('Averaged FFT Magnitude', fontsize=12)
    plt.title(title)
    plt.grid(True, which='both', alpha=0.5)
    plt.ylim(bottom=10**-11)
    pdf.savefig()
    plt.show()
    plt.close()

    return freqs, avg_fft_magnitude
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
with PdfPages('Averaged_FFT_plots_.pdf') as pdf:
    for i, dframe in enumerate(car_on_music_dataframes):
        mean_shifted_column = f"Signal{i+1} Zero Mean (m)"
        title_ = f"Averaged Frequency Spectrum - Measurement {i+1}"
        freqs, avg_mags = plot_averaged_fft(dframe, mean_shifted_column, sr=25000, segment_size=8096, title=title_)

##### Power Spectral Density With Welch's Method:
- Change the scaling parameter in welch() from 'density' to 'spectrum' to plot a squared magnitude spectrum instead

In [ ]:
def welch_psd(data, column, sr, title, segment_size, window='hann'):
    """data: pandas DataFrame
    column: relevant data column from data, str
    sr: Sampling rate in Hz
    title: title for PSD plot, str
    segment_size: length of each segment
    window: desired window to use"""
    
    overlap = segment_size//2 # number of points to overlap between segments, here is 50% overlap
    freqs, psd = welch(data[column].values, fs=sr, nperseg=segment_size, noverlap=overlap, window=window, scaling='density')

    # Plot:
    plt.figure(figsize=(12,6))
    #plt.plot(freqs,psd)
    plt.loglog(freqs, psd)
    plt.xlabel("Frequency (Hz)")
    plt.ylabel("PSD ($m^2$/ Hz)")
    plt.title(title)
    plt.grid(True, which='both', alpha=0.5)
    plt.ylim(bottom=10**-22)
    pdf.savefig()
    plt.show()
    plt.close()

    return freqs, psd
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
with PdfPages('PSD_plots_.pdf') as pdf:
    for i, dframe in enumerate(car_on_music_dataframes):
        mean_shifted_column = f"Signal{i+1} Zero Mean (m)"
        w_title = f"Power Spectral Density - Measurement {i+1}"
        w_freqs, w_psd = welch_psd(dframe, mean_shifted_column, sr=25000, title=w_title, segment_size=8096)

##### Continuous Wavelet Transform
- The Morlet wavelet, given by $\psi(t)=\exp(\frac{-t^2}{2})\cos(5t)$, is used for the wavelet transform
- Note: may want to reduce look of horizontal bands near bottom of plots in future

In [ ]:
with PdfPages('CWT_plots_.pdf') as pdf:
    for i, dframe in enumerate(car_on_music_dataframes):
        # Setup Data:
        sampling_rate= 25000 # Hz
        sig_col= f"Signal{i+1} Zero Mean (m)"
        signal = apply_hann_window(dframe, sig_col)
        time = dframe[f"Time{i+1} (s)"]
    
        downsample_factor = 10 # downsample to speed up computation
        signal_ds = scipy.signal.decimate(signal, downsample_factor)
        sampling_rate_ds = sampling_rate // downsample_factor
        time_ds = time[::downsample_factor]
        print(f"Sample rate after downsampling: {sampling_rate_ds} Hz")
        print(f"Nyquist: {sampling_rate_ds / 2} Hz")
    
        # Setup Resolution:
        nv = 32 # number of voices/wavelets per octave
        # Perform CWT:
        Wx, scales = sq.cwt(signal_ds, 'morlet', nv=nv, fs=sampling_rate_ds)
        frequencies = scale_to_freq(scales, 'morlet', len(signal_ds), fs=sampling_rate_ds)
    
        # Max frequency should not exceed Nyquist --> limit freq range based on this:
        freq_mask = (frequencies >=5) & (frequencies <= sampling_rate_ds / 2)
        freqs_plot = frequencies[freq_mask]
        Wx_masked = Wx[freq_mask, :]         
    
        # Time downsample for display
        time_skip = 50
        Wx_plot = np.log10(np.abs(Wx_masked[:, ::time_skip])+ 1e-10) # add small epsilon to prevent log error
        time_display = time_ds[::time_skip]
    
        # Option: Define the "floor" for the data
        # e.g. anything quieter than threshold_val will be treated as "Not a Number" (NaN), which renders as transparent/white.
        threshold_val = -9.5 # note: vmin should be close to threshold value
        Wx_plot[Wx_plot < threshold_val] = np.nan
    
        # Plot:
        plt.figure(figsize=(14, 6))
        mesh = plt.pcolormesh(time_display, freqs_plot, Wx_plot, cmap='jet', shading='auto', vmin=threshold_val)
        plt.yscale('log')
        plt.ylabel('Frequency (Hz)')
        plt.xlabel('Time (s)')
        plt.title(f'Car On With Music - Measurement {i+1}')
        # Add colorbar
        plt.colorbar(mesh, label='log\u2081\u2080(Magnitude)')
        pdf.savefig()
        plt.show()
        plt.close()

***
### Convert Cell Outputs to PDF (Old Version):
- can currently download as .html file, which when opened in a browser can be saved as a pdf using print (Ctrl+P) -> save to pdf <br>
    - using code to directly download as .pdf file is not working currently
    - if opened with Jupyter Notebook, can save full file as .pdf directly by going to File -> Save and Export Notebook As -> PDF

In [ ]:
# Save notebook as .html file
# def save_notebook(notebook_name, output_name=None):
#     """Save Jupyter notebook outputs and markdown to .html file (excludes code cells)
#     notebook_name : str, name of the notebook file (e.g., 'notebook.ipynb')
#     output_name : str, name for the output file (optional). If None, uses the notebook name with .html extension"""
    
#     # Set output file name
#     if output_name is None:
#         output_name = notebook_name.replace('.ipynb', '.html')
    
#     # Check if notebook file exists
#     if not os.path.exists(notebook_name):
#         raise FileNotFoundError(f"Notebook file '{notebook_name}' not found")
            
#     result = subprocess.run(['jupyter', 'nbconvert', '--to', 'html', '--no-input',  # Exclude code cells
#     notebook_name, '--output', output_name.replace('.html', '')], capture_output=True, text=True)
            
#     if result.returncode == 0: # return code of zero indicates successful execution 
#         print(f"Created HTML file: {output_name}")
#         print("Open this in your browser and use Ctrl+P --> Save as PDF")
#         print("(The HTML file contains only outputs and markdown, no code cells)")
#     else:
#         print("Error:")
#         print(result.stderr) # standard error

# # Example usage:
# # save_notebook('notebook.ipynb')
# # Or specify custom output name:
# # save_notebook('notebook.ipynb', 'output_notebook.html')
# save_notebook('Car_On_Vibrometer_Plots.ipynb', 'Car_On_Vibrometer_Plots.html')